In [ ]:
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = ""
os.environ["LANGSMITH_PROJECT"] = "run-traces"
os.environ["OPENAI_API_KEY"] = ""
os.environ["XAI_API_KEY"]=""

In [ ]:
import json
import re
import time
from anthropic import APIError
import operator
from datetime import datetime
from langchain.schema import HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_xai import ChatXAI
from langchain_anthropic import ChatAnthropic
from langchain_core.tracers.context import tracing_v2_enabled
from langsmith import traceable
import uuid
from langsmith import Client
from langsmith.run_helpers import get_current_run_tree
from langchain_community.callbacks import get_openai_callback
import pandas as pd
import ast
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from IPython.display import Image, display
from __future__ import annotations
from typing import Annotated, TypedDict, List
from langgraph.graph.message import add_messages
from typing import Annotated
from operator import itemgetter  # For taking the last value
from langchain_core.messages import BaseMessage  # Add this import
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    AIMessage,
    SystemMessage
)
from langgraph.graph.message import add_messages
from typing import Annotated, TypedDict


class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]  # ✅ Use LangGraph's message reducer
    phase: Annotated[str, lambda _, x: x]
    max_retries: Annotated[int, lambda _, x: x]
    current_agent: Annotated[str, lambda _, x: x]
    user_profile: dict
    step_idx: int
    same_step_turns: int
    selected_recipe: dict | None
    chef_agent: 'ChefAgent'
    trainee_agent: 'TraineeAgent' 
    user_query: str | None
    clarified_topics: list[str]


 # Changed from CookingState
def supervisor_router(state: State) -> str:
    if state.get("max_retries", 0) >= 3:
        return END
    
    phase = state["phase"]
    last_agent = state.get("current_agent")
    
    if phase == "chef_turn" and last_agent == "chef":
        state["max_retries"] += 1
    elif phase == "trainee_turn" and last_agent == "trainee":
        state["max_retries"] += 1
    else:
        state["max_retries"] = 0

    # Clear routing logic
    return {
        "introduction": "chef",
        "recipe_selection": "chef", 
        "chef_turn": "trainee",
        "trainee_turn": "chef",
        "done": END
    }.get(phase, END)




def supervisor_node(state: State):
    phase = state["phase"]
    
    # Hard exit after 3 retries
    if state.get("retries", 0) >= 3:
        state["phase"] = "done"
        return state
        
    # Existing phase transitions
    if phase == "introduction":
        state["phase"] = "recipe_selection"
    elif phase == "recipe_selection":
        state["phase"] = "chef_turn" if state.get("selected_recipe") else "recipe_selection"
    elif phase == "chef_turn":
        state["phase"] = "trainee_turn" if state.get("selected_recipe") else "recipe_selection"
    elif phase == "trainee_turn":
        state["phase"] = "chef_turn" if state.get("selected_recipe") else "recipe_selection"
        
    return state





        
def steps_from_recipe(state: State) -> list[str]:
    """Extract steps from recipe instructions"""
    if not state.get("selected_recipe"):
        return []
    
    instructions = state["selected_recipe"].get("Instructions", "")
    if not instructions:
        return []
    
    # ✅ Simplified parsing that always returns at least 1 step
    return [s.strip() for s in re.split(r'\n+|\d+\.', instructions) if s.strip()]

# In chef_node:
def chef_node(state: State):
    phase = state["phase"]
    try:
        if phase == "introduction":
            response = state["chef_agent"].respond(
                state["messages"],
                "Introduce yourself as ChefAI, your friendly cooking assistant. Ask the user what they'd like to do: find a specific recipe, get similar recipes, or find recipes by ingredients."
            )
            state["messages"].append(AIMessage(content=response))
            state["phase"] = "recipe_selection"
            return state
        # General validation
        if phase in ["chef_turn", "ingredient_check"] and not state.get("selected_recipe"):
            response = "No recipe selected yet. Please select a recipe to continue."
            state["messages"].append(AIMessage(content=response))
            state["phase"] = "recipe_selection"
            return state

        if phase == "recipe_selection":
            # Add error handling for recipe selection
            try:
                response = state["chef_agent"].select_recipe(state["user_query"], state)
                state["messages"].append(AIMessage(content=response))
                
                if "Matching recipes:" in response or "Similar recipes:" in response:
                    state["phase"] = "user_select_recipe"
                else:
                    state["phase"] = "recipe_selection"  # Stay in selection phase
                    
                # Increment retry counter on errors
                if "Error" in response:
                    state["retries"] = state.get("retries", 0) + 1
                    
                return state
            except Exception as e:
                state["messages"].append(AIMessage(content=f"Error: {str(e)}"))
                state["phase"] = "recipe_selection"
                state["retries"] = state.get("retries", 0) + 1
                return state


        elif phase == "chef_turn":
            steps = steps_from_recipe(state)
            current_step_idx = state.get("step_idx", 0)
            if current_step_idx >= len(steps):
                response = "Recipe completed! Great job!"
                state["phase"] = "done"
            else:
                step_text = steps[current_step_idx]
                response = f"Step {current_step_idx + 1}: {step_text}"
            state["messages"].append(AIMessage(content=response))
            return state

        elif phase == "ingredient_check":
            try:
                ingredients = ast.literal_eval(state["selected_recipe"]['Cleaned_Ingredients'])
                msg = f"Required ingredients for {state['selected_recipe']['Title']}:\n"
                msg += "\n".join(f"- {i}" for i in ingredients)
                state["messages"].append(AIMessage(content=msg))
                state["phase"] = "trainee_confirm_ingredients"
            except Exception as e:
                print(f"Error parsing ingredients: {e}")
                state["phase"] = "recipe_selection"
            return state

        # Add other phases as needed

    except Exception as e:
        # General error handling
        state["messages"].append(AIMessage(content=f"Chef encountered an error: {e}"))
        state["phase"] = "recipe_selection"
        return state

    return state


def trainee_node(state: State):
    phase = state["phase"]
    if phase == "trainee_turn" and not state.get("selected_recipe"):
        state["messages"].append(HumanMessage(content="No recipe selected."))
        state["phase"] = "recipe_selection"
        return state
    if phase in ("trainee_choose_recipe", "user_select_recipe"):  # <--- handle both
        # Extract recipe list from last Chef message
        recipe_list = extract_recipes_from_message(state["messages"][-1].content)
        chosen_recipe_name = state["trainee_agent"].choose_recipe(recipe_list, state)
        recipe = get_recipe_by_name(chosen_recipe_name, recipes_df)
        if recipe:
            state["selected_recipe"] = recipe
            allergies = state["user_profile"].get("allergies", [])
            if contains_allergen(recipe['Cleaned_Ingredients'], allergies):
                state["phase"] = "allergy_warning"
            else:
                state["phase"] = "ingredient_check"
        else:
            # If no recipe found, fallback to recipe_selection
            state["phase"] = "recipe_selection"
        return state  # <-- THIS LINE IS CRITICAL

    elif phase == "trainee_confirm_ingredients":
        response = state["trainee_agent"].confirm_ingredients(state)
        state["messages"].append(HumanMessage(content=response))
        if "yes" in response.lower():
            state["phase"] = "chef_turn"
        else:
            state["phase"] = "recipe_selection"
        return state


    elif phase == "trainee_turn":
        # Get the last chef message (the current step)
        chef_msgs = [msg for msg in state["messages"] if isinstance(msg, AIMessage)]
        last_chef_msg = chef_msgs[-1].content if chef_msgs else ""
        steps = steps_from_recipe(state)
        response = state["trainee_agent"].generate_response(last_chef_msg, len(steps))
        state["messages"].append(HumanMessage(content=response))
        # Only increment if the trainee says "next"
        if "next" in response.lower():
            state["step_idx"] += 1
            state["same_step_turns"] = 0
        else:
            state["same_step_turns"] = state.get("same_step_turns", 0) + 1
            # Optionally, force step progression after too many questions
            if state["same_step_turns"] > 2:
                state["step_idx"] += 1
                state["same_step_turns"] = 0
        state["phase"] = "chef_turn"
        return state



def extract_recipes_from_message(message):
    # Extract recipe names from the chef's message
    # e.g., parse after "Matching recipes:" or "Similar recipes:"
    return [r.strip() for r in re.split(r',|\n', message.split(':', 1)[-1]) if r.strip()]



def build_cooking_graph():
    graph_builder = StateGraph(State)
    
    # Only 3 nodes: supervisor, chef, trainee
    graph_builder.add_node("supervisor", supervisor_node)
    graph_builder.add_node("chef", chef_node)
    graph_builder.add_node("trainee", trainee_node)
    
    # Start with supervisor
    graph_builder.add_edge(START, "supervisor")
    
    # Single conditional edge routing
    graph_builder.add_conditional_edges(
        "supervisor",
        supervisor_router,
        {
            "chef": "chef",
            "trainee": "trainee",
            END: END
        }
    )
    
    # Agents always return to supervisor
    graph_builder.add_edge("chef", "supervisor")
    graph_builder.add_edge("trainee", "supervisor")
    
    return graph_builder.compile()






class IntentDetection(BaseModel):
    """Identify user intent and extract key details"""
    intent_type: str = Field(..., description="Type of request: 'specific_recipe', 'similar_recipes', 'ingredient_search'")
    target_recipe: str | None = Field(None, description="Recipe name if specified")
    ingredients: list[str] | None = Field(None, description="List of ingredients if provided")
recipes_df = pd.read_csv("13k-recipes.csv")

def classify_step_complexity(step_text):
    # Simple heuristic: long steps or those with multiple actions are 'complex'
    if len(step_text.split()) > 30:
        return "complex"
    if any(word in step_text.lower() for word in ["meanwhile", "until", "while", "simultaneously", "at the same time", "then"]):
        return "complex"
    # You can add more rules as needed
    return "simple"

def clarification_node(state: State):
    # Present the step in more detail, allow multiple questions
    last_chef_msg = [m.content for m in state["messages"] if isinstance(m, AIMessage)][-1]
    print(f"\nChef (Clarification): {last_chef_msg}")
    user_input = input("Trainee (ask a question or type 'next' to proceed): ").strip().lower()
    state["messages"].append(HumanMessage(content=user_input))
    if user_input == "next":
        state["clarified"] = True
        state["phase"] = "chef"
    else:
        # Optionally, answer the question or break down the step further
        # You can use your chef_agent to generate a more detailed explanation
        answer = state["chef_agent"].respond(state["messages"], f"Explain this step in more detail: {last_chef_msg}")
        state["messages"].append(AIMessage(content=answer))
        state["clarified"] = False
        state["phase"] = "clarification"
    return state


client = Client()
runs = client.list_runs(project_name="run-traces")

# for run in runs:
#     print("Run name:", run.name)
#     print("Start time:", run.start_time)
#     print("End time:", run.end_time)
#     print("Duration (seconds):", (run.end_time - run.start_time).total_seconds())

class ChefAgent:
    def __init__(self, job_id, recipes_df, trainee_experience_log=None):
        # self.llm = ChatXAI(model="",
        #                     xai_api_key=os.environ["XAI_API_KEY"]
        # )
        self.llm = ChatOpenAI(model="gpt-4.1-mini", openai_api_key=os.environ["OPENAI_API_KEY"])
        self.structured_llm = ChatOpenAI(model="gpt-4.1-mini", openai_api_key=os.environ["OPENAI_API_KEY"]).with_structured_output(IntentDetection)
        self.job_id = job_id
        self.token_cost_log = []    
        self.recipes_df = recipes_df
        self.trainee_experience_log = trainee_experience_log
        self.system_message = self.build_system_message()

    def detect_intent(self, user_query: str) -> IntentDetection:
        prompt = f"""Analyze this cooking-related query:
        {user_query}

        Classify the intent:
        - 'specific_recipe' if asking for a particular dish
        - 'similar_recipes' if requesting variations
        - 'ingredient_search' if listing ingredients they have
        
        Extract recipe names or ingredients as needed."""
        
        return self.structured_llm.invoke(prompt)
    
    def get_current_step(self, state: State) -> str:
        """Safe step retrieval with error handling"""
        steps = steps_from_recipe(state)
        idx = state.get("step_idx", 0)
        
        if not steps:
            return "No steps available for current recipe"
        if idx >= len(steps):
            return "Recipe completed!"
            
        return steps[idx]
    

    def select_recipe(self, user_query, state):
        try:
            # 1. Detect intent
            intent = self.detect_intent(user_query)
            
            # 2. Handle different intents
            if intent.intent_type == "specific_recipe":
                recipe = get_recipe_by_name(intent.target_recipe, recipes_df)
                if not recipe:
                    return "Recipe not found. Please try another."
                return f"Selected recipe: {recipe['Title']}"
            
            elif intent.intent_type == "similar_recipes":
                similar = get_similar_recipes(intent.target_recipe, recipes_df)
                if similar.empty:
                    return "No similar recipes found."
                return f"Similar recipes: {', '.join(similar['Title'].tolist())}"
            
            elif intent.intent_type == "ingredient_search":
                matches = get_recipes_by_ingredients(intent.ingredients, recipes_df)
                if matches.empty:
                    return "No recipes match those ingredients."
                return f"Matching recipes: {', '.join(matches['Title'].tolist())}"
            
            else:
                return "Please clarify your request"
                
        except Exception as e:
            print(f"Recipe selection error: {e}")
            return "Error finding recipes. Please try again."

    def build_system_message(self):
        experience_level = self.trainee_experience_log.get('experience_level', 'unknown') if self.trainee_experience_log else 'unknown'
        allergies = self.trainee_experience_log.get('allergies', []) if self.trainee_experience_log else []
        preferred_cuisine = self.trainee_experience_log.get('preferred_cuisine', 'any') if self.trainee_experience_log else 'any'
        notes = self.trainee_experience_log.get('notes', '') if self.trainee_experience_log else ''

        allergy_str = ", ".join(allergies) if allergies else "none"

        experience_info = f"""
    User Profile:
    - Experience level: {experience_level}
    - Allergies: {allergy_str}
    - Preferred cuisine: {preferred_cuisine}
    - Notes: {notes}

    IMPORTANT INSTRUCTIONS:
    - NEVER suggest or proceed with any recipe or step that contains any of the user's allergens: {allergy_str}.
    - If the user requests a recipe with an allergen, gently warn them and suggest safe alternatives.
    - ALWAYS adapt your explanations to the user's experience level. For beginners, explain techniques, tools, and terminology in simple terms, and offer encouragement.
    - If the user has never performed a technique before (see notes), provide extra explanation and support when that technique arises.
    - Before each step, check if the user is comfortable and ready to proceed.
    """

        base_instructions = """
    Please follow these instructions carefully:

    1. Introduction:
    - Introduce yourself as ChefAI.
    - Briefly explain that you're here to help with recipe preparation.

    2. Recipe Confirmation:
    - Confirm the recipe name with the user.

    3. Ingredient Recall:
    - List all the ingredients required for the recipe.
    - Ask the user if they have all the ingredients ready.

    4. Step-by-Step Guidance:
    - Provide instructions one step at a time.
    - After each step, wait for the user to say "next" or ask a question before proceeding.
    - If the user asks a question, answer it thoroughly before continuing with the recipe steps.

    5. Completion:
    - When all steps are complete, congratulate the user and ask if they need any final advice.

    Before each interaction with the user, wrap your analysis in <recipe_analysis> tags to process the recipe, organize your thoughts, and prepare your response. In this analysis:
    - Break down the recipe into key components: ingredients, equipment needed, and cooking techniques.
    - Identify potential challenges or areas where users might need extra guidance.
    - Plan out how to present each step in a clear and concise manner.
    This will help you provide accurate and helpful guidance without revealing all instructions at once.

    Example interaction format:

    ChefAI: [Introduction and recipe confirmation]
    User: [Response]
    ChefAI: [List ingredients and ask if ready]
    User: [Response]
    ChefAI: [Provide first step]
    User: next
    ChefAI: [Provide next step]
    User: [Question]
    ChefAI: [Answer question]
    User: next
    ... (continue until recipe is complete)

    Remember to maintain a friendly and encouraging tone throughout the interaction. Begin your first response now by introducing yourself and confirming the recipe.
    """
        return SystemMessage(content=experience_info + base_instructions)


    @traceable(run_type="chain", name="Chef Response", metadata={"role": "chef"})
    def respond(self, conversation, prompt):
        run = get_current_run_tree()
        if run:
            run.metadata["job_id"] = self.job_id
            run.metadata["run_id"] = str(run.id)
        messages = [self.system_message] + conversation + [HumanMessage(content=prompt)]
        max_retries = 5
        base_delay = 1

        for attempt in range(max_retries):
            try:
                with get_openai_callback() as cb:
                    response = self.llm.invoke(messages).content.strip()
                # Log token and cost info
                self.token_cost_log.append({
                    "prompt_tokens": cb.prompt_tokens,
                    "completion_tokens": cb.completion_tokens,
                    "total_tokens": cb.total_tokens,
                    "cost": cb.total_cost,
                    "timestamp": datetime.now().isoformat(),
                    "prompt": prompt
                })
                update_running_totals(self.token_cost_log)
                print(f"ChefAgent LLM call: {cb.total_tokens} tokens, ${cb.total_cost:.5f}")
                return response
            except APIError as e:
                if "overloaded_error" in str(e):
                    delay = base_delay * (2 ** attempt)
                    print(f"OverloadedError: Retrying in {delay}s (attempt {attempt+1}/{max_retries})")
                    time.sleep(delay)
                else:
                    raise
        raise Exception("Max retries exceeded")


class TraineeAgent:
    def __init__(self, job_id, trainee_experience_log=None, conversation_mode="mixed"):
        self.llm = ChatOpenAI(model="gpt-4.1-mini", openai_api_key=os.environ["OPENAI_API_KEY"])
        self.job_id = job_id
        self.current_step = 0
        self.recipe_data = None
        self.conversation_mode = conversation_mode
        self.token_cost_log = []
        self.trainee_experience_log = trainee_experience_log
        
    @traceable(run_type="chain", name="Trainee Response", metadata={"role": "trainee"})
    def generate_response(self, chef_message, num_steps):
        experience_level = self.trainee_experience_log.get("experience_level", "beginner").lower()
        notes = self.trainee_experience_log.get("notes", "")

        # Adjust question-asking behavior based on experience
        if experience_level == "advanced":
            question_instruction = (
                "ONLY ask a question if this step is ambiguous or unusually challenging for an expert cook. "
                "If everything is clear, say 'next'. Do NOT ask about basic techniques or substitutions."
            )
        elif experience_level == "intermediate":
            question_instruction = (
                "Ask a question if you are unsure about a technique or ingredient. "
                "Otherwise, say 'next'."
            )
        else:  # beginner
            question_instruction = (
                "If you have any doubt about the technique, ingredient, or process in this step, ask ONE SHORT, direct question. "
                "Otherwise, say 'next'."
            )

        # Optionally, use notes for further customization (e.g., never roasted a chicken)
        if notes and "never" in notes.lower():
            question_instruction += (
                f" You have noted: {notes}. If this step involves something you have never done, ask for extra explanation."
            )

        prompt = f"""You're following a recipe with {num_steps} steps. Current step: {self.current_step+1}.
    Last instruction: {chef_message}
    {question_instruction}
    """
        with get_openai_callback() as cb:
            response = self.llm.invoke([HumanMessage(content=prompt)])
            self.token_cost_log.append({
                "prompt_tokens": cb.prompt_tokens,
                "completion_tokens": cb.completion_tokens,
                "total_tokens": cb.total_tokens,
                "cost": cb.total_cost,
                "timestamp": datetime.now().isoformat(),
                "prompt": prompt
            })
            update_running_totals(self.token_cost_log)
            print(f"TraineeAgent LLM call: {cb.total_tokens} tokens, ${cb.total_cost:.5f}")
            return response.content.strip().lower()
        
    def confirm_ingredients(self, state: State) -> str:
        """TraineeAgent confirms ingredient readiness automatically"""
        try:
            # Get ingredients from last message
            ingredients_msg = state["messages"][-1].content
            allergies = state["user_profile"].get("allergies", [])
            
            # Check for allergens automatically
            has_allergen = any(allergen.lower() in ingredients_msg.lower() for allergen in allergies)
            
            if has_allergen:
                return f"I notice this recipe contains {', '.join(allergies)} which I'm allergic to. Can we find an alternative?"
            else:
                return "Yes, I have all the ingredients ready to proceed."
                
        except Exception as e:
            print(f"Error in confirm_ingredients: {e}")
            return "Yes, let's proceed with the recipe."
        
    def choose_recipe(self, recipe_list: list[str], state: State) -> str:
        """Automatically choose a recipe based on preferences"""
        try:
            preferences = state["user_profile"]
            preferred_cuisine = preferences.get("preferred_cuisine", "").lower()
            allergies = preferences.get("allergies", [])
            
            # Simple logic: pick first recipe that matches preferences and avoids allergens
            for recipe in recipe_list:
                # Check for preferred cuisine
                if preferred_cuisine and preferred_cuisine in recipe.lower():
                    # Check for allergens (basic check)
                    if not any(allergen.lower() in recipe.lower() for allergen in allergies):
                        return recipe
            
            # Fallback: return first recipe
            return recipe_list[0] if recipe_list else ""
            
        except Exception as e:
            print(f"Error choosing recipe: {e}")
            return recipe_list[0] if recipe_list else ""


def parse_steps(recipe_text):
    steps = [s.strip() for s in re.split(r'\n{2,}|\n', recipe_text) if s.strip()]
    return steps

def update_running_totals(log_list):
    total_tokens = 0
    total_cost = 0
    for entry in log_list:
        total_tokens += entry["total_tokens"]
        total_cost += entry["cost"]
        entry["cumulative_tokens"] = total_tokens
        entry["cumulative_cost"] = total_cost

def append_with_accrual(conversation, message, chef, trainee, combined_accrual):
    conversation.append(message)
    chef_cum = chef.token_cost_log[-1] if chef.token_cost_log else {"cumulative_tokens": 0, "cumulative_cost": 0}
    trainee_cum = trainee.token_cost_log[-1] if trainee.token_cost_log else {"cumulative_tokens": 0, "cumulative_cost": 0}
    combined_accrual.append({
        "chef_cumulative_tokens": chef_cum["cumulative_tokens"],
        "chef_cumulative_cost": chef_cum["cumulative_cost"],
        "trainee_cumulative_tokens": trainee_cum["cumulative_tokens"],
        "trainee_cumulative_cost": trainee_cum["cumulative_cost"],
        "overall_cumulative_tokens": chef_cum["cumulative_tokens"] + trainee_cum["cumulative_tokens"],
        "overall_cumulative_cost": chef_cum["cumulative_cost"] + trainee_cum["cumulative_cost"]
    })

# # Example experience log
# trainee_experience_log = {
#     "experience_level": "beginner",  # Options: 'beginner', 'intermediate', 'advanced'
#     "preferred_cuisine": "Italian",
#     "allergies": ["nuts"]
# }


def get_recipe_by_name(name: str, df: pd.DataFrame) -> dict | None:
    matches = df[df['Title'].str.lower().str.contains(name.lower(), na=False)]
    return matches.iloc[0].to_dict() if not matches.empty else None

def get_similar_recipes(name: str, df: pd.DataFrame) -> pd.DataFrame:
    return df[df['Title'].str.lower().str.contains(name.lower(), na=False)].head(5)

def get_recipes_by_ingredients(ingredients: list[str], df: pd.DataFrame) -> pd.DataFrame:
    mask = df['Cleaned_Ingredients'].apply(
        lambda x: all(ing.lower() in str(x).lower() for ing in ingredients)
    )
    return df[mask].head(5)


def contains_allergen(ingredients, allergies):
    try:
        ingredients_list = ast.literal_eval(ingredients)
    except Exception:
        ingredients_list = [ingredients]
    return any(allergen.lower() in " ".join(ingredients_list).lower() for allergen in allergies)

def visualize_graph(graph):
    """Visualize the LangGraph structure using built-in methods"""
    display(Image(graph.get_graph().draw_mermaid_png()))
    print(graph.get_graph().draw_mermaid())
    return graph.get_graph().draw_mermaid()

    
    # Initialize state
def automated_cooking_session(job_id, trainee_experience_log=None):
    chef = ChefAgent(job_id, recipes_df, trainee_experience_log)
    trainee = TraineeAgent(job_id, trainee_experience_log=trainee_experience_log)
    system_message = chef.system_message
    conversation = []
    combined_accrual = []
    interaction_log = []

    # Build and visualize the graph structure BEFORE user input
    graph = build_cooking_graph()
    # print("\n" + visualize_graph(graph) + "\n")  # <-- This will display the graph immediately

    # Now prompt the user
    user_query = input("What would you like to cook? ")
    initial_state = {
        "messages": [system_message],  # System prompt is first!
        "phase": "introduction",
        "user_profile": trainee_experience_log,
        "step_idx": 0,
        "retries": 0,
        "same_step_turns": 0,
        "selected_recipe": None,
        "chef_agent": chef,
        "trainee_agent": trainee,
        "user_query": user_query,
        "clarified_topics": []
    }

    
    # Build and compile graph
    graph = build_cooking_graph()
    
    # Increase recursion limit and add debugging
    config = {"recursion_limit": 200}  # From 25 to 100
    
    # Print graph structure
    print("\n" + visualize_graph(graph) + "\n")
    final_state = None

    # Run conversation loop with debugging
    for output in graph.stream(initial_state, config=config, stream_mode="values"):
        print(f"\n=== Current State ===")
        print(f"Phase: {output.get('phase', 'unknown')}")
        print(f"Step: {output.get('step_idx', 0)+1}")
        
        # Print last chef message
        chef_msgs = [msg for msg in output.get("messages", []) if isinstance(msg, AIMessage)]
        if chef_msgs:
            print(f"\nChef: {chef_msgs[-1].content}")
        
        # Print last trainee message
        trainee_msgs = [msg for msg in output.get("messages", []) if isinstance(msg, HumanMessage)] 
        if trainee_msgs:
            print(f"Trainee: {trainee_msgs[-1].content}")
            
        # Check for completion
        if output.get("phase") == "done":
            final_state = output
            break

    if final_state is not None:
        save_conversation_to_json(job_id, final_state, chef, trainee)
    else:
        print("No completed conversation to save.")



    print("ChefAgent system prompt:\n", chef.system_message.content)

    # User selects query type
    # user_query = input("What would you like to cook? ")
    intent = chef.detect_intent(user_query)  # Now returns IntentDetection
    
    # Access fields directly
    print(f"Detected intent: {intent.intent_type}")
    print(f"Target recipe: {intent.target_recipe}")
    print(f"Ingredients: {intent.ingredients}")

    if intent.intent_type == "specific_recipe":
        selected_recipe = get_recipe_by_name(intent.target_recipe, recipes_df)
        if selected_recipe is None:
            print("No recipe found.")
            return
    elif intent.intent_type == "similar_recipes":
        similar = get_similar_recipes(intent.target_recipe, recipes_df)
        if similar.empty:
            print("No similar recipes found.")
            return
        print("Similar recipes found:")
        print(similar['Title'].to_list())
        user_choice = input("Which recipe do you want? ")
        selected_recipe = get_recipe_by_name(user_choice, recipes_df)
        if selected_recipe is None:
            print("No recipe found.")
            return
    elif intent.intent_type == "ingredient_search":
        matches = get_recipes_by_ingredients(intent.ingredients, recipes_df)
        if matches.empty:
            print("No recipes found for those ingredients.")
            return
        print("Recipes you can make:")
        print(matches['Title'].to_list())
        user_choice = input("Which recipe do you want? ")
        selected_recipe = get_recipe_by_name(user_choice, recipes_df)
        if selected_recipe is None:
            print("No recipe found.")
            return
    else:
        print("Please clarify your request")
        return


    # Now proceed with your step-by-step logic using selected_recipe
    initial_state["selected_recipe"] = selected_recipe
    recipe_text = selected_recipe['Instructions']
    steps = parse_steps(recipe_text)
    ingredients = selected_recipe['Cleaned_Ingredients']

    try:
        ingredients_list = ast.literal_eval(ingredients)
        if not isinstance(ingredients_list, list):
            ingredients_list = [ingredients]  # fallback: treat as single string
    except Exception:
        ingredients_list = [ingredients]  # fallback: treat as single string

    # 2. Chef lists ingredients and asks if trainee is ready
    ingredients_message = (
        f"Here are the ingredients you'll need for '{selected_recipe['Title']}':\n"
        + "\n".join(f"- {item}" for item in ingredients_list)
        + "\nDo you have all these ingredients ready? (yes/no)"
    )
    print("\nChefAI:", ingredients_message)
    append_with_accrual(conversation, AIMessage(content=ingredients_message), chef, trainee, combined_accrual)

    # 3. Wait for trainee confirmation before proceeding
    trainee_response = input("\nTrainee: ").strip().lower()
    while trainee_response not in ["yes", "y"]:
        print("ChefAI: Please gather all the ingredients before we start. Let me know when you're ready!")
        append_with_accrual(conversation, AIMessage(content="Please gather all the ingredients before we start. Let me know when you're ready!"), chef, trainee, combined_accrual)
        trainee_response = input("\nTrainee: ").strip().lower()

    # 5. Step-by-step guidance
    steps = parse_steps(recipe_text)
    for idx, step in enumerate(steps):
    # Chef's turn
        chef_step = chef.respond(conversation, f"Step {idx+1}: {step}\nExplain...")
        print(f"\nChef (Step {idx+1}):", chef_step)
        append_with_accrual(conversation, AIMessage(content=chef_step), chef, trainee, combined_accrual)

        # Trainee's turn
        trainee.current_step = idx
        trainee_msg = trainee.generate_response(chef_step, len(steps))
        print("\nTrainee:", trainee_msg)
        append_with_accrual(conversation, HumanMessage(content=trainee_msg), chef, trainee, combined_accrual)

    # Log interaction metrics
    interaction_log.append({
        "step": idx+1,
        "chef_tokens": chef.token_cost_log[-1]["total_tokens"],
        "trainee_tokens": trainee.token_cost_log[-1]["total_tokens"],
        "total_cost": chef.token_cost_log[-1]["cost"] + trainee.token_cost_log[-1]["cost"],
        "cumulative_tokens": chef.token_cost_log[-1]["cumulative_tokens"] + trainee.token_cost_log[-1]["cumulative_tokens"],
        "cumulative_cost": chef.token_cost_log[-1]["cumulative_cost"] + trainee.token_cost_log[-1]["cumulative_cost"]
    })

    # 6. End message
    chef_end = chef.respond(conversation, "Say cooking is complete and offer congratulations.")
    print("\nChef:", chef_end)
    conversation.append(AIMessage(content=chef_end))

    # 7. Save conversation to JSON

    

    conversation_log = []
    for i, m in enumerate(conversation):
        entry = {
            "role": "trainee" if isinstance(m, HumanMessage) else "chef",
            "content": m.content
        }
        if i < len(combined_accrual):
            entry.update(combined_accrual[i])
        conversation_log.append(entry)


    chef_total_tokens = sum(entry["total_tokens"] for entry in chef.token_cost_log)
    chef_total_cost = sum(entry["cost"] for entry in chef.token_cost_log)

    # Aggregate Trainee stats
    trainee_total_tokens = sum(entry["total_tokens"] for entry in trainee.token_cost_log)
    trainee_total_cost = sum(entry["cost"] for entry in trainee.token_cost_log)

    # Overall totals
    overall_total_tokens = chef_total_tokens + trainee_total_tokens
    overall_total_cost = chef_total_cost + trainee_total_cost

    # Print results
    print("\n=== Token and Cost Summary ===")
    print(f"ChefAgent:    {chef_total_tokens} tokens, ${chef_total_cost:.6f}")
    print(f"TraineeAgent: {trainee_total_tokens} tokens, ${trainee_total_cost:.6f}")
    print(f"TOTAL:        {overall_total_tokens} tokens, ${overall_total_cost:.6f}")


    last_chef = chef.token_cost_log[-1]
    print(f"ChefAgent running total: {last_chef['cumulative_tokens']} tokens, ${last_chef['cumulative_cost']:.6f}")

    last_trainee = trainee.token_cost_log[-1]
    print(f"TraineeAgent running total: {last_trainee['cumulative_tokens']} tokens, ${last_trainee['cumulative_cost']:.6f}")

    if 'combined_accrual' not in locals():
        combined_accrual = []

    # Get last chef and trainee cumulative totals (or 0 if none yet)
    chef_cum = chef.token_cost_log[-1] if chef.token_cost_log else {"cumulative_tokens": 0, "cumulative_cost": 0}
    trainee_cum = trainee.token_cost_log[-1] if trainee.token_cost_log else {"cumulative_tokens": 0, "cumulative_cost": 0}

    combined_accrual.append({
        "turn": len(combined_accrual) + 1,
        "chef_cumulative_tokens": chef_cum["cumulative_tokens"],
        "chef_cumulative_cost": chef_cum["cumulative_cost"],
        "trainee_cumulative_tokens": trainee_cum["cumulative_tokens"],
        "trainee_cumulative_cost": trainee_cum["cumulative_cost"],
        "overall_cumulative_tokens": chef_cum["cumulative_tokens"] + trainee_cum["cumulative_tokens"],
        "overall_cumulative_cost": chef_cum["cumulative_cost"] + trainee_cum["cumulative_cost"]
    })


    # Prepare summary for saving
    token_cost_summary = {
        "chef_total_tokens": chef_total_tokens,
        "chef_total_cost": chef_total_cost,
        "trainee_total_tokens": trainee_total_tokens,
        "trainee_total_cost": trainee_total_cost,
        "overall_total_tokens": overall_total_tokens,
        "overall_total_cost": overall_total_cost,
        "chef_turns": chef.token_cost_log,
        "trainee_turns": trainee.token_cost_log
    }

    with open(f"cooking_session_{job_id}_full_log.json", "w", encoding="utf-8") as f:
        json.dump(
            {
                "job_id": job_id,
                "conversation": conversation_log,
                "metrics": {
                    # "interactions": interaction_log,  # New per-interaction data
                    "chef_turns": chef.token_cost_log,
                    "trainee_turns": trainee.token_cost_log,
                }
            },
            f, indent=2, ensure_ascii=False
        )


import json
from datetime import datetime

def save_conversation_to_json(job_id, state, chef, trainee):
    conversation_log = []
    messages = state.get("messages", [])
    for i, m in enumerate(messages):
        entry = {
            "role": "trainee" if isinstance(m, HumanMessage) else "chef",
            "content": m.content,
            "timestamp": datetime.now().isoformat()
        }
        # Optionally, add token/cost info if available
        conversation_log.append(entry)

    # Optionally, add summary stats
    chef_total_tokens = sum(entry["total_tokens"] for entry in chef.token_cost_log)
    chef_total_cost = sum(entry["cost"] for entry in chef.token_cost_log)
    trainee_total_tokens = sum(entry["total_tokens"] for entry in trainee.token_cost_log)
    trainee_total_cost = sum(entry["cost"] for entry in trainee.token_cost_log)
    overall_total_tokens = chef_total_tokens + trainee_total_tokens
    overall_total_cost = chef_total_cost + trainee_total_cost

    summary = {
        "chef_total_tokens": chef_total_tokens,
        "chef_total_cost": chef_total_cost,
        "trainee_total_tokens": trainee_total_tokens,
        "trainee_total_cost": trainee_total_cost,
        "overall_total_tokens": overall_total_tokens,
        "overall_total_cost": overall_total_cost
    }

    # Save as JSON file with unique job_id
    with open(f"cooking_session_{job_id}_full_log.json", "w", encoding="utf-8") as f:
        json.dump(
            {
                "job_id": job_id,
                "conversation": conversation_log,
                "metrics": summary
            },
            f, indent=2, ensure_ascii=False
        )


# Usage with your recipe
if __name__ == "__main__":
    job_id = str(uuid.uuid4())
    trainee_experience_log = {
        "experience_level": "Advanced",
        "preferred_cuisine": "Italian",
        "allergies": ["milk"],
        "notes": "Has never roasted a chicken"
    }
    with tracing_v2_enabled(project_name="run-traces", tags=[f"job_id:{job_id}"]):
        automated_cooking_session(job_id, trainee_experience_log=trainee_experience_log)

ValueError: 'chef_agent' is already being used as a state key